In [12]:
import os, json, uuid, time
from datetime import datetime
import os, json, glob, numpy as np, pandas as pd
from PIL import Image
IMAGES_DIR = "img/images_org"          
MASKS_DIR  = "img/outputs_sam2"           
DEPTH_DIR  = "img/zoe_outputs"    

DEPTH_IS_METRIC = True               # set False if ZoeDepth is relative
PIXELS_PER_CM_GLOBAL = None          # e.g., 15.3 if you know it; else None

DENSITY_LABELS_CSV = None            # optional CSV: image,mask_index,label

KNOWN_PLATE_DIAMETER_CM = None       # e.g., 27.0 (cm) if using a standard plate


RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"Week5_{RUN_ID}"
os.makedirs(RUN_DIR, exist_ok=True)
LOG_PATH = os.path.join(RUN_DIR, "run_log.jsonl")
CONFIG = {}
# Optional: if you know total meal weight (grams) per image later, you can map here:
TOTAL_MEAL_WEIGHT_BY_IMAGE = {
    # "image1.jpeg": 520.0,
    # "image2.jpeg": 480.0,
}

def log(step, **kw):
    with open(LOG_PATH, "a") as f:
        f.write(json.dumps({"t": time.time(), "step": step, **kw}) + "\n")
    print(f"[LOG] {step} ::", {k:v for k,v in kw.items()})

print("Run dir:", RUN_DIR)


Run dir: Week5_20251015_150931


In [13]:
# === Cell 2: Loaders (images, masks, depth) ===
def list_images(images_dir):
    SUP = (".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp")
    names = [n for n in os.listdir(images_dir) if n.lower().endswith(SUP)]
    names.sort()
    return names

def load_masks_for(stem):
    """
    Preferred: MASKS_DIR/<stem>.npz with key 'masks' [N,H,W] (0/1).
    Fallback:  MASKS_DIR/<stem>/*.png (0/255) -> booleans.
    """
    npz = os.path.join(MASKS_DIR, f"{stem}.npz")
    if os.path.exists(npz):
        data = np.load(npz)
        key = "masks" if "masks" in data else list(data.keys())[0]
        stack = data[key]
        masks = [(stack[i] > 0).astype(np.uint8) for i in range(stack.shape[0])]
        return masks

    sub = os.path.join(MASKS_DIR, stem)
    files = sorted(glob.glob(os.path.join(sub, "*.png")))
    if files:
        ms = []
        for p in files:
            arr = np.array(Image.open(p))
            if arr.ndim==3: arr = arr[...,0]
            ms.append((arr > 0).astype(np.uint8))
        return ms

    raise FileNotFoundError(f"No masks for '{stem}' in {MASKS_DIR}")

def load_depth_for(stem):
    """
    Matches names like: image1_overlay_depth_m.npy (metric), image1_depth.png, etc.
    Priority:
      *_depth*m*.npy  (metric)
      *_depth*.npy
      *{stem}*.npy
      *_depth*.png (16-bit mm -> m)
      *{stem}*.png
    Returns float32 [H,W].
    """
    # metric npy first
    cands = sorted(glob.glob(os.path.join(DEPTH_DIR, f"*{stem}*depth*m*.npy")))
    if cands:
        arr = np.load(cands[0]).astype(np.float32)
        print(f"[depth] matched metric npy: {os.path.basename(cands[0])}  shape={arr.shape}")
        return arr
    # generic depth npy
    cands = sorted(glob.glob(os.path.join(DEPTH_DIR, f"*{stem}*depth*.npy")))
    if cands:
        arr = np.load(cands[0]).astype(np.float32)
        print(f"[depth] matched depth npy:  {os.path.basename(cands[0])}  shape={arr.shape}")
        return arr
    # any npy with stem
    cands = sorted(glob.glob(os.path.join(DEPTH_DIR, f"*{stem}*.npy")))
    if cands:
        arr = np.load(cands[0]).astype(np.float32)
        print(f"[depth] matched generic npy: {os.path.basename(cands[0])}  shape={arr.shape}")
        return arr
    # png depths
    for pat in (f"*{stem}*depth*.png", f"*{stem}*.png"):
        cands = sorted(glob.glob(os.path.join(DEPTH_DIR, pat)))
        if cands:
            arr = np.array(Image.open(cands[0])).astype(np.float32)
            if arr.max() > 1000:  # likely millimeters
                arr = arr / 1000.0
                print(f"[depth] matched png (mm->m): {os.path.basename(cands[0])}  shape={arr.shape}")
            else:
                print(f"[depth] matched png (raw):   {os.path.basename(cands[0])}  shape={arr.shape}")
            return arr
    raise FileNotFoundError(f"No depth found for '{stem}' in {DEPTH_DIR}")


In [14]:
# === Cell 3: Geometry helpers ===
import cv2
import numpy as np

def resize_mask_to_depth(mask_u8: np.ndarray, depth_hw):
    """Resize a binary mask to match depth shape using nearest-neighbor."""
    H, W = depth_hw
    if mask_u8.ndim == 3:
        mask_u8 = mask_u8[...,0]
    mask_u8 = (mask_u8 > 0).astype(np.uint8)
    if mask_u8.shape == (H, W):
        return mask_u8
    return cv2.resize(mask_u8, (W, H), interpolation=cv2.INTER_NEAREST).astype(np.uint8)

def table_depth_from_bbox_ring(depth, mask, ring=12):
    H, W = depth.shape
    ys, xs = np.where(mask > 0)
    if xs.size == 0:
        return float(np.median(depth))
    x0, y0, x1, y1 = xs.min(), ys.min(), xs.max()+1, ys.max()+1
    X0, Y0 = max(0, x0-ring), max(0, y0-ring)
    X1, Y1 = min(W, x1+ring), min(H, y1+ring)
    outer = depth[Y0:Y1, X0:X1]
    ring_mask = np.ones_like(outer, dtype=bool)
    ring_mask[(y0-Y0):(y1-Y0), (x0-X0):(x1-X0)] = False
    vals = outer[ring_mask]
    if vals.size < 50: vals = depth
    return float(np.median(vals))

def volume_cm3_from_mask(depth_m, mask_u8, pixels_per_cm, dep_is_metric=True):
    """
    If pixels_per_cm is None or depth isn't metric, returns (value, 'arb').
    Otherwise returns (cm^3 value, 'cm^3').
    """
    if mask_u8.sum() == 0:
        return 0.0, ("cm^3" if (dep_is_metric and pixels_per_cm) else "arb")
    t = table_depth_from_bbox_ring(depth_m, mask_u8, ring=12)
    h = (t - depth_m).clip(min=0)  # meters if Zoe is metric
    if (not dep_is_metric) or (pixels_per_cm is None):
        return float(h[mask_u8 > 0].sum()), "arb"
    height_cm = h[mask_u8 > 0] * 100.0
    cm_per_px = 1.0 / float(pixels_per_cm)
    vol_cm3 = float(height_cm.sum() * (cm_per_px**2))
    return vol_cm3, "cm^3"

# Densities (extend as needed)
DENSITY_G_PER_CM3 = {
    "rice": 0.85, "pasta": 0.60, "bread": 0.27, "chicken_cooked": 1.05,
    "beef_cooked": 1.10, "pork_cooked": 1.05, "fish_cooked": 1.02,
    "potato_cooked": 0.75, "salad": 0.20, "soup": 1.00, "curry": 1.00,
    "tofu": 0.90, "egg_boiled": 1.03, "beans": 0.77, "default": 0.85,
}

def grams_from_volume(vol_value, vol_unit, label_hint=None):
    label = label_hint if (label_hint in DENSITY_G_PER_CM3) else "default"
    rho = DENSITY_G_PER_CM3[label]
    if vol_unit == "cm^3":
        return float(vol_value * rho), rho, label
    return float("nan"), rho, label

def compute_pixels_per_cm(img_name, depth_shape, plate_bbox=None):
    cfg = CONFIG.get(img_name, {})
    if "pixels_per_cm" in cfg and cfg["pixels_per_cm"]:
        return float(cfg["pixels_per_cm"])
    if PIXELS_PER_CM_GLOBAL:
        return float(PIXELS_PER_CM_GLOBAL)
    if KNOWN_PLATE_DIAMETER_CM and plate_bbox:
        x0,y0,x1,y1 = plate_bbox
        px = max(x1-x0, y1-y0)
        return px / KNOWN_PLATE_DIAMETER_CM
    return None


In [15]:
# === Cell 4: Run on all images ===
from IPython.display import display

rows = []
images = list_images(IMAGES_DIR)
print("Found images:", images)

for img_name in images:
    stem, _ = os.path.splitext(img_name)
    depth = load_depth_for(stem)
    masks = load_masks_for(stem)

    # Resize masks to depth resolution (CRITICAL)
    H, W = depth.shape
    masks = [resize_mask_to_depth(m, (H, W)) for m in masks]

    # Pick scale (pixels per cm) if available
    plate_bbox = CONFIG.get(img_name, {}).get("plate_bbox")
    px_per_cm = compute_pixels_per_cm(img_name, (H, W), plate_bbox=plate_bbox)

    print(f"[INFO] {img_name}: depth {depth.shape}, masks {len(masks)}, px/cm={px_per_cm}")

    # Compute volumes/weights; also compute relative mass fractions
    rel_mass_scores = []
    per_item = []

    for mi, m in enumerate(masks):
        if m.sum() == 0:
            continue
        vol, unit = volume_cm3_from_mask(depth, m, px_per_cm, dep_is_metric=DEPTH_IS_METRIC)

        # Optional: plug your per-mask label hints here (else 'default')
        label_hint = None
        grams, rho, used_label = grams_from_volume(vol, unit, label_hint)

        # Relative mass proxy (works even when unit=='arb')
        rel_mass = (vol if unit=="arb" else vol) * rho
        rel_mass_scores.append(rel_mass)

        per_item.append({
            "image": img_name,
            "mask_index": mi,
            "pixels_per_cm": px_per_cm,
            "volume_value": vol,
            "volume_unit": unit,             # 'cm^3' if scaled, else 'arb'
            "density_label": used_label,
            "density_g_per_cm3": rho,
            "weight_g": grams,               # NaN if no scale
            "mask_pixels": int(m.sum()),
            "rel_mass_score": rel_mass,      # for fractions
        })

    # Normalize to mass fractions per image
    total_rel_mass = float(np.sum(rel_mass_scores)) if rel_mass_scores else 0.0
    total_known_weight = TOTAL_MEAL_WEIGHT_BY_IMAGE.get(img_name)

    for r in per_item:
        frac = (r["rel_mass_score"] / total_rel_mass) if total_rel_mass > 0 else 0.0
        r["mass_fraction"] = float(frac)
        if total_known_weight is not None and (np.isfinite(frac)):
            r["weight_g_from_fraction"] = float(frac * total_known_weight)
        else:
            r["weight_g_from_fraction"] = None

    rows.extend(per_item)

df = pd.DataFrame(rows).sort_values(["image","mask_index"]).reset_index(drop=True)
display(df)


Found images: ['image1.jpeg', 'image2.jpeg', 'images.jpeg']
[depth] matched metric npy: image1_overlay_depth_m.npy  shape=(183, 275)
[INFO] image1.jpeg: depth (183, 275), masks 3, px/cm=None
[depth] matched metric npy: image2_overlay_depth_m.npy  shape=(168, 300)
[INFO] image2.jpeg: depth (168, 300), masks 5, px/cm=None
[depth] matched metric npy: images_overlay_depth_m.npy  shape=(198, 254)
[INFO] images.jpeg: depth (198, 254), masks 7, px/cm=None


,image,mask_index,pixels_per_cm,volume_value,volume_unit,density_label,density_g_per_cm3,weight_g,mask_pixels,rel_mass_score,mass_fraction,weight_g_from_fraction
0,image1.jpeg,0,None,2691.509277,arb,default,0.85,NaN,17003,2287.782886,0.701811,None
1,image1.jpeg,1,None,1026.628662,arb,default,0.85,NaN,8201,872.634363,0.267693,None
2,image1.jpeg,2,None,116.953415,arb,default,0.85,NaN,4721,99.410403,0.030496,None
3,image2.jpeg,0,None,94.119148,arb,default,0.85,NaN,29655,80.001276,0.022811,None
4,image2.jpeg,1,None,3590.455566,arb,default,0.85,NaN,13217,3051.887231,0.870209,None
5,image2.jpeg,2,None,127.674683,arb,default,0.85,NaN,1370,108.523480,0.030944,None
6,image2.jpeg,3,None,241.556549,arb,default,0.85,NaN,1266,205.323067,0.058545,None
7,image2.jpeg,4,None,72.162445,arb,default,0.85,NaN,824,61.338078,0.017490,None
8,images.jpeg,0,None,733.747375,arb,default,0.85,NaN,17984,623.685269,0.248144,None
9,images.jpeg,1,None,514.815857,arb,default,0.85,NaN,17129,437.593478,0.174104,None
